# Predicting arrears next month
Follows `protocol.yaml`: temporal holdout (`holdout.csv`), 5-fold walk-forward CV on training rows only, selection by highest mean CV ROC AUC, holdout scored once.

**Feature decision:** `final_notice_sent` is recorded *in the billing cycle that followed* the billing month, i.e. after prediction time and a near-proxy for the outcome (86.5% arrears when 1 vs 1.4% when 0). It is excluded as leakage. `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
is_hold=np.zeros(len(df),bool); is_hold[hold_idx]=True
train=df[~is_hold]; hold=df[is_hold]
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM
print(len(train),len(hold),train.billing_month.max(),hold.billing_month.min())

In [ ]:
def prep():
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler(),NUM)])
# all candidates named before any is scored
candidates={
 'baseline_prior': Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1': Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1': Pipeline([('p',prep()),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'rf_depth6': Pipeline([('p',prep()),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hgb_small': Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
# walk-forward CV over months, training rows only: 5 expanding-window folds
months=np.sort(train.billing_month.unique())
chunks=np.array_split(months,6)   # first chunk = initial train; next 5 = validation blocks
folds=[]
for k in range(1,6):
    tr=np.where(train.billing_month.isin(np.concatenate(chunks[:k])))[0]
    va=np.where(train.billing_month.isin(chunks[k]))[0]
    folds.append((tr,va))
    print(k,len(tr),len(va),chunks[k][0],chunks[k][-1])
cv_scores={}
for name,m in candidates.items():
    s=[]
    for tr,va in folds:
        mm=clone(m).fit(train.iloc[tr][FEATURES],train.iloc[tr][TARGET])
        s.append(float(roc_auc_score(train.iloc[va][TARGET],mm.predict_proba(train.iloc[va][FEATURES])[:,1])))
    cv_scores[name]=s
    print(f'{name:16s} mean={np.mean(s):.4f}',np.round(s,4))
chosen=max(cv_scores,key=lambda n:np.mean(cv_scores[n]))
print('chosen:',chosen)

In [ ]:
# refit on all training rows, score once on holdout
final=clone(candidates[chosen]).fit(train[FEATURES],train[TARGET])
holdout_auc=float(roc_auc_score(hold[TARGET],final.predict_proba(hold[FEATURES])[:,1]))
print('holdout roc_auc',holdout_auc)

In [ ]:
train_pos=np.where(~is_hold)[0]  # 0-based positions in data.csv
evidence={
 'features':FEATURES,
 'candidates':list(candidates),
 'cv_folds':[[train_pos[tr].tolist(),train_pos[va].tolist()] for tr,va in folds],
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':hold_idx.tolist(),
 'holdout_scores':{chosen:holdout_auc},
}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the printed CV and holdout ROC AUC above. `final_notice_sent` was deliberately not used because it is not known at prediction time.